中断通过在图节点的任何位置调用 interrupt() 函数来实现

In [1]:
from langgraph.types import interrupt
from langgraph.graph import MessagesState

class State(MessagesState):
    approved: bool

def approval_node(state: State):
    # Pause and ask for approval
    approved = interrupt("Do you approve this action?")

    # When you resume, Command(resume=...) returns that value here
    return {"approved": approved}

In [2]:
from langgraph.graph import END, START, StateGraph
from langgraph.checkpoint.memory import InMemorySaver


graph = StateGraph(State)
graph.add_node("approval", approval_node)
graph.add_edge(START, "approval")
graph.add_edge("approval", END)
graph = graph.compile(checkpointer=InMemorySaver())

In [3]:
from langgraph.types import Command

# Initial run - hits the interrupt and pauses
# thread_id is the persistent pointer (stores a stable ID in production)
config = {"configurable": {"thread_id": "thread-1"}}
result = graph.invoke({"messages": [{"role": "user", "content": "data"}]}, config=config, version="v2") # type: ignore

# result is a GraphOutput with .value and .interrupts
# .interrupts contains the payloads passed to interrupt()
print(result.interrupts)

(Interrupt(value='Do you approve this action?', id='410d4d971cf3154b2f7a4b13dfffa327'),)


In [4]:
result.interrupts[0].value

'Do you approve this action?'

通过重新调用图使用 Command 恢复执行，该值随后将成为节点内 interrupt() 调用的返回值。

In [5]:
# Resume with the human's response
# The resume payload becomes the return value of interrupt() inside the node
response = graph.invoke(Command(resume=True), config=config, version="v2") # type: ignore

In [6]:
response.value

{'messages': [HumanMessage(content='data', additional_kwargs={}, response_metadata={}, id='0a0ccfe4-0ef7-4613-88e4-1370fae1fff1')],
 'approved': True}

#### 流式输出中的中断

In [7]:
config = {'configurable': {'thread_id': 'thread-2'}}
state = {'messages': [{'role': 'user', 'content': 'data'}]}
for chunk in graph.stream( # type: ignore
    state, # type: ignore
    config=config, # type: ignore
    stream_mode='updates',
    version='v2',
):  # type: ignore
    if chunk['type'] == 'updates':
        if '__interrupt__' in chunk['data']:
            print("Received interrupt:", chunk['data']['__interrupt__'])
            state = Command(resume=True)

for chunk in graph.stream( # type: ignore
    state, # type: ignore
    config=config, # type: ignore
    stream_mode='updates',
    version='v2',
):  # type: ignore
    if chunk['type'] == 'updates':
        print("Received update:", chunk['data'])

Received interrupt: (Interrupt(value='Do you approve this action?', id='1d586161fc521cc6620a0c9ae09d011f'),)
Received update: {'approval': {'approved': True}}
